# Preprocess analysis

Load `preprocess_metrics.csv`, apply cheap filters, and export a new Python item list for the heavy homog screener.

In [45]:
from pathlib import Path

REPO = Path.cwd()
if not (REPO / 'skin_homog' / 'screener_preprocess' / 'preprocess_metrics.csv').exists():
    REPO = Path('C:/Roman/skins_roundtrip_v1')

INPUT_CSV = REPO / 'skin_homog' / 'screener_preprocess' / 'preprocess_metrics.csv'
OUTPUT_LIST = REPO / 'lists' / 'skins_normal_filtered1.py'

BASE_MIN = 3.0
BASE_MAX = 100.0
N_LISTINGS_MIN = 15
REQUIRE_CAP_HIT = False
AVG_DISCOUNT_MIN = -0.05
AVG_DISCOUNT_MAX = None
MEDIAN_DISCOUNT_MIN = None
DISCOUNT_SAMPLE_N_MIN = 3


In [46]:
import pandas as pd

df = pd.read_csv(INPUT_CSV)
print(f'rows={len(df)} from {INPUT_CSV}')
display(df.head())
display(df[['base_price', 'n_listings', 'avg_discount', 'median_discount']].describe(include='all'))


rows=6597 from C:\Roman\skins_roundtrip_v1\skin_homog\screener_preprocess\preprocess_metrics.csv


,item,status,error,sort_by,collected_at_utc,base_price,currency,n_listings,cap_hit,discount_sample_n,avg_discount,median_discount,avg_ask,avg_predicted,key_trace
0,AK-47 | Aphrodite (Battle-Scarred),ok,NaN,best_deal,2026-04-17T00:32:24.854338+00:00,16.57,NaN,50,True,8,-0.103877,-0.104406,18.29125,16.57000,1/1
1,AK-47 | Aphrodite (Factory New),ok,NaN,best_deal,2026-04-17T00:32:28.693179+00:00,470.00,NaN,50,True,8,-0.012346,-0.009245,475.84875,470.04625,1/1
2,AK-47 | Aphrodite (Field-Tested),ok,NaN,best_deal,2026-04-17T00:32:31.877135+00:00,20.96,NaN,50,True,8,-0.088823,-0.107347,24.83500,22.89500,1/1
3,AK-47 | Aphrodite (Minimal Wear),ok,NaN,best_deal,2026-04-17T00:32:36.971279+00:00,45.92,NaN,50,True,8,-0.009884,-0.005009,46.38375,45.93000,1/1
4,AK-47 | Aphrodite (Well-Worn),ok,NaN,best_deal,2026-04-17T00:32:40.145863+00:00,16.29,NaN,50,True,8,-0.078257,-0.060160,17.70250,16.41125,1/1


,base_price,n_listings,avg_discount,median_discount
count,6469.000000,6597.000000,6469.000000,6469.000000
mean,83.488255,29.608458,-5.177806,-1.981736
std,538.934273,19.410340,61.106379,30.407438
min,0.010000,0.000000,-2536.286014,-1428.071429
25%,0.430000,10.000000,-0.834165,-0.600000
50%,3.700000,32.000000,-0.163690,-0.134615
75%,23.810000,50.000000,-0.027827,-0.025265
max,16108.880000,50.000000,0.294174,0.300643


In [47]:
mask_status = df['status'].fillna('') == 'ok'
mask_base_notna = df['base_price'].notna()
mask_base_min = df['base_price'] >= BASE_MIN
mask_base_max = df['base_price'] <= BASE_MAX
mask_listings = df['n_listings'].fillna(0) >= N_LISTINGS_MIN
mask_sample = df['discount_sample_n'].fillna(0) >= DISCOUNT_SAMPLE_N_MIN
mask_avg_notna = df['avg_discount'].notna()
mask_avg_min = df['avg_discount'] >= AVG_DISCOUNT_MIN
mask_avg_max = pd.Series(True, index=df.index) if AVG_DISCOUNT_MAX is None else (df['avg_discount'] <= AVG_DISCOUNT_MAX)
mask_median = pd.Series(True, index=df.index) if MEDIAN_DISCOUNT_MIN is None else (df['median_discount'].notna() & (df['median_discount'] >= MEDIAN_DISCOUNT_MIN))
mask_cap = pd.Series(True, index=df.index) if not REQUIRE_CAP_HIT else df['cap_hit'].fillna(False).astype(bool)

filters = [
    ('status_ok', mask_status),
    ('base_notna', mask_base_notna),
    ('base_min', mask_base_min),
    ('base_max', mask_base_max),
    ('n_listings_min', mask_listings),
    ('discount_sample_n_min', mask_sample),
    ('avg_discount_notna', mask_avg_notna),
    ('avg_discount_min', mask_avg_min),
    ('avg_discount_max', mask_avg_max),
    ('median_discount_min', mask_median),
    ('require_cap_hit', mask_cap),
]

total = len(df)
print('standalone filter impact:')
for name, mask in filters:
    kept = int(mask.sum())
    print(f'  {name}: keeps {kept} / {total}, drops {total - kept}')

print('')
print('sequential filter impact:')
running = pd.Series(True, index=df.index)
prev_kept = int(running.sum())
for name, mask in filters:
    running &= mask
    kept = int(running.sum())
    print(f'  after {name}: {kept} / {total} (step drop {prev_kept - kept})')
    prev_kept = kept

keep = running
filtered = df.loc[keep].sort_values(['avg_discount', 'n_listings', 'item'], ascending=[False, False, True]).reset_index(drop=True)
print('')
print(f'kept {len(filtered)} / {len(df)} items')
display(filtered.head(5))


standalone filter impact:
  status_ok: keeps 6597 / 6597, drops 0
  base_notna: keeps 6469 / 6597, drops 128
  base_min: keeps 3448 / 6597, drops 3149
  base_max: keeps 5778 / 6597, drops 819
  n_listings_min: keeps 4395 / 6597, drops 2202
  discount_sample_n_min: keeps 6079 / 6597, drops 518
  avg_discount_notna: keeps 6469 / 6597, drops 128
  avg_discount_min: keeps 2035 / 6597, drops 4562
  avg_discount_max: keeps 6597 / 6597, drops 0
  median_discount_min: keeps 6597 / 6597, drops 0
  require_cap_hit: keeps 6597 / 6597, drops 0

sequential filter impact:
  after status_ok: 6597 / 6597 (step drop 0)
  after base_notna: 6469 / 6597 (step drop 128)
  after base_min: 3448 / 6597 (step drop 3021)
  after base_max: 2757 / 6597 (step drop 691)
  after n_listings_min: 1709 / 6597 (step drop 1048)
  after discount_sample_n_min: 1709 / 6597 (step drop 0)
  after avg_discount_notna: 1709 / 6597 (step drop 0)
  after avg_discount_min: 1124 / 6597 (step drop 585)
  after avg_discount_max: 1124 

,item,status,error,sort_by,collected_at_utc,base_price,currency,n_listings,cap_hit,discount_sample_n,avg_discount,median_discount,avg_ask,avg_predicted,key_trace
0,P90 | Ancient Earth (Factory New),ok,NaN,best_deal,2026-04-18T14:34:34.614115+00:00,6.25,NaN,50,True,8,0.294174,0.300643,7.26125,10.28125,1/1
1,Dual Berettas | Switch Board (Factory New),ok,NaN,best_deal,2026-04-17T09:49:25.337211+00:00,12.29,NaN,45,False,8,0.267631,0.260581,14.11875,19.26875,1/1->1/1
2,P90 | Traction (Factory New),ok,NaN,best_deal,2026-04-18T16:36:07.554321+00:00,7.56,NaN,36,False,8,0.249481,0.245840,8.20250,10.93500,1/1->1/1
3,MAC-10 | Whitefish (Factory New),ok,NaN,best_deal,2026-04-18T03:09:48.533204+00:00,8.29,NaN,38,False,8,0.240692,0.239797,8.68375,11.43375,1/1->1/1
4,Tec-9 | Phoenix Chalk (Factory New),ok,NaN,best_deal,2026-04-18T20:26:55.193047+00:00,24.40,NaN,50,True,8,0.219894,0.212894,31.66125,40.50625,1/1


In [48]:
import json

items = filtered['item'].dropna().astype(str).tolist()
filters = {
    'INPUT_CSV': str(INPUT_CSV),
    'BASE_MIN': BASE_MIN,
    'BASE_MAX': BASE_MAX,
    'N_LISTINGS_MIN': N_LISTINGS_MIN,
    'REQUIRE_CAP_HIT': REQUIRE_CAP_HIT,
    'AVG_DISCOUNT_MIN': AVG_DISCOUNT_MIN,
    'AVG_DISCOUNT_MAX': AVG_DISCOUNT_MAX,
    'MEDIAN_DISCOUNT_MIN': MEDIAN_DISCOUNT_MIN,
    'DISCOUNT_SAMPLE_N_MIN': DISCOUNT_SAMPLE_N_MIN,
    'N_OUTPUT_ITEMS': len(items),
}

OUTPUT_LIST.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_LIST.open('w', encoding='utf-8') as f:
    f.write('# Auto-generated from preprocess screener\n')
    for key, value in filters.items():
        f.write(f'# {key} = {value!r}\n')
    f.write('\nITEMS = [\n')
    for item in items:
        f.write(f'    {json.dumps(item, ensure_ascii=False)},\n')
    f.write(']\n')

print(f'saved {len(items)} items -> {OUTPUT_LIST}')


saved 1124 items -> C:\Roman\skins_roundtrip_v1\lists\skins_normal_filtered1.py
